# İHA Video Arama — Colab denemesi

Bu defter **gerçek GPU'da** pipeline'ı uçtan uca çalıştırır ve
proje-ozeti.md §8'in en kritik doğrulanmamış varsayımını ölçer:
embedding hızı (§8 40x gerçek-zaman varsayıyor, ölçülmedi).

## Colab'da NE ÇALIŞIR

| Bileşen | Durum |
|---|---|
| Proxy üretimi (ffmpeg + NVDEC/NVENC) | ✅ |
| Embedding (Qwen3-VL-Embedding-2B) | ✅ **asıl ölçüm burada** |
| YOLO26 görsel alanlar | ✅ |
| Qdrant (gömülü mod) | ✅ işlevsel, ⚠️ performans ölçümü geçersiz |
| Nesne deposu (yerel dizin) | ✅ MinIO/Docker gerekmiyor |
| Sorgu hattı + filtre gevşetme | ✅ |
| vLLM (yapısal ayrıştırma) | ⚠️ mümkün ama VRAM'i bölüyor — 9. bölüm |
| Temporal + Kafka orkestrasyon | ❌ Docker yok |
| Ölçek testi (100K+) | ❌ oturum/disk sınırı |

## İki önemli uyarı

**1. Qdrant gömülü mod gerçek motor değil.** `qdrant-client`'in saf Python
implementasyonu; Rust HNSW yerine tam (exact) arama yapıyor. Sonuçlar
işlevsel olarak geçerli (hatta Recall daha yüksek — yaklaşıklık yok) ama
**gecikme ölçümleri anlamsız**. Gerçek HNSW için 10. bölümdeki Qdrant
ikili dosyası yolunu kullanın.

**2. Colab'ın T4'ü bf16 Tensor Core içermiyor** (compute 7.5). Kod bunu
algılayıp fp16'ya geçiyor. Ölçtüğünüz hız **RTX 4060'ı temsil etmez**
(4060 compute 8.9, bf16 doğal). A100/L4 seçebiliyorsanız daha temsili olur.

---

## Kullanım kuralları

- **Hücreleri sırayla çalıştırın.** 2. hücre kodu günceller, 3. hücre ortamı
  kurar; sonrakiler bunlara dayanıyor.
- **Ortam değişkenini `set_env(...)` ile değiştirin**, doğrudan `os.environ`
  ile değil. `common/config.py` env'i import anında okuyor; `set_env`
  ayrıca modülü yeniden yüklüyor. Aksi halde süreç-içi çağrılar eski
  değeri görür.
- **Modeller bir kez yüklenir.** 6. bölümden itibaren her şey süreç-içi
  çalışıyor, `!python -m ...` değil — alt süreç kullanmak video başına
  ~1 dakikayı modeli yeniden yüklemeye harcıyordu.
- **Qdrant istemcisini `close()` etmeyin.** Gömülü mod dosya kilidi
  kullanıyor; `get_client()` önbellekli, ikinci bir istemci açmak hata verir.

## Bu defterde daha önce çıkan hatalar (hepsi düzeltildi)

| Belirti | Sebep |
|---|---|
| `Connection refused :9000` | MinIO gerekmiyor — artık `LOCAL_STORAGE_PATH` kullanılıyor |
| Eski kod çalışıyor, hata alakasız yerde | Sessiz `git pull` — 2. hücre artık çalışan sürümü basıyor |
| `No module named 'minio'` | `config` env ayarlanmadan önce import edilmişti — `set_env` çözüyor |
| `moov atom not found` | Video dosyası eksik/bozuk — 5. hücre artık önceden yakalıyor |
| `'OutStream' object has no attribute 'reconfigure'` | `sys.stdout.reconfigure` Jupyter'da yok |

Takıldığınızda ilk komut: `!python -m scripts.check_env`

## 1. GPU kontrolü

Çalıştırma türü → Donanım hızlandırıcı → **GPU** seçili olmalı.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!free -g | head -2
!df -h /content | tail -1

## 2. Depo + bağımlılıklar

Colab'da torch zaten CUDA'lı geldiği için `requirements.txt`'i olduğu gibi
kurmuyoruz (mevcut torch'u bozabilir). Sadece eksikleri kuruyoruz.

In [ ]:
import pathlib, subprocess, sys

REPO = pathlib.Path('/content/VideoAnalysis')

# Sessiz git komutlari kullanmiyoruz: onceki surumde `git clone -q ...
# 2>/dev/null || git pull -q` vardi, pull sessizce basarisiz olunca ESKI KOD
# calismaya devam etti ve hata cok sonra alakasiz bir yerde patladi.
if REPO.exists():
    print('Depo mevcut, uzak surumle esitleniyor...')
    # Colab'daki klon tek kullanimlik - yerel degisiklik korunmasi gerekmiyor.
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
    print(subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO,
                         capture_output=True, text=True).stdout.strip())
else:
    r = subprocess.run(['git', 'clone',
                        'https://github.com/ykyking1/VideoAnalysis.git', str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

head = subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('\nCalisan surum:', head)

# Surum kontrolu DOSYA OKUYARAK - config'i BURADA IMPORT ETMIYORUZ.
# common/config.py ortam degiskenlerini import aninda okuyor; bir sonraki
# hucrede LOCAL_STORAGE_PATH ayarlanacak. Simdi import edersek modul eski
# (bos) degerlerle onbellege girer ve surec-ici cagrilarda MinIO'ya duser.
config_src = (REPO / 'common' / 'config.py').read_text(encoding='utf-8')
assert 'LOCAL_STORAGE_PATH' in config_src, (
    'ESKI KOD! git pull calismamis - yukaridaki "Calisan surum" satirini kontrol edin.')

%cd /content/VideoAnalysis

# Colab'in CUDA'li torch'una DOKUNMUYORUZ - sadece eksik paketler.
# qwen-vl-utils>=0.0.14 kritik: eskisi SESSIZCE bozuk embedding uretiyor.
# minio'yu KURMUYORUZ: yerel dizin arka ucu kullanacagiz.
!pip install -q "transformers>=4.57" "qwen-vl-utils>=0.0.14" accelerate \
    qdrant-client ultralytics opencv-python-headless temporalio \
    pysolar shapely 2>&1 | tail -3

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Kod guncel, hazir. Sonraki hucrede ortam degiskenleri ayarlanacak.')

## 3. Ortam yapılandırması

Colab'da Docker yok. İki servisi de Docker'sız çalıştırıyoruz:

- **Qdrant** → gömülü mod (`QDRANT_LOCAL_PATH`)
- **Nesne deposu** → yerel dizin (`LOCAL_STORAGE_PATH`), MinIO'ya gerek yok

> MinIO ikilisini arka planda çalıştırmayı denemiştik ama sessizce ölüp
> hatayı çok sonra `Connection refused` olarak gösteriyordu. Pipeline'ın
> nesne deposundan tek ihtiyacı "dosya koy / dosya al" olduğu için yerel
> dizin aynı işi görüyor ve bir kırılma noktası eksiliyor.
>
> **Üretimde MinIO kullanın** — dosya sistemi arka ucu tek makineye bağlı,
> dağıtık worker'lar paylaşımlı nesne deposu gerektiriyor.

In [ ]:
import os, sys, importlib

def set_env(**kwargs):
    """Ortam degiskenini ayarlar VE common.config'i yeniden yukler.

    common/config.py env'i IMPORT ANINDA okuyor. Notebook'ta bir degiskeni
    (ornegin EMBEDDING_BATCH_SIZE) sonradan degistirirseniz, config zaten
    yuklenmis oldugu icin surec-ici cagrilar ESKI degeri gorur. Bu yardimci
    o tuzagi kapatiyor - notebook boyunca env degistirmek icin hep bunu
    kullanin, dogrudan os.environ'a yazmayin.

    Tum modullerimiz `from common import config` (modul referansi) kullandigi
    icin reload degerleri her yere yayilir."""
    for k, v in kwargs.items():
        os.environ[k] = str(v)
    if 'common.config' in sys.modules:
        importlib.reload(sys.modules['common.config'])

# Docker'siz calisan iki arka uc
set_env(
    QDRANT_LOCAL_PATH='/content/qdrant_data',   # Qdrant gomulu mod
    LOCAL_STORAGE_PATH='/content/storage',      # nesne deposu = yerel dizin
    EMBEDDING_BATCH_SIZE=8,                     # T4 16GB icin baslangic
    EMBEDDING_DTYPE='auto',                     # T4 (compute 7.5) -> fp16
    CAPTION_ENABLED='false',                    # vLLM yok (bkz. 9. bolum)
)

from common import config
from common.minio_client import backend_name
assert config.LOCAL_STORAGE_PATH, 'LOCAL_STORAGE_PATH okunmadi'
print('Nesne deposu :', backend_name())
print('Qdrant       : gomulu ->', config.QDRANT_LOCAL_PATH)
print('Batch        :', config.EMBEDDING_BATCH_SIZE)

!python -m scripts.init_storage --skip-postgres

## 4. Ortam doğrulaması

`torch ... CPU-only` ya da `qwen-vl-utils < 0.0.14` görürseniz **durun** —
ikisi de çökmeden sessizce bozuyor.

In [ ]:
!python -m scripts.check_env

## 5. Veri

Videolarınızı `/content/videos/` altına koyun. Seçenekler:
- **Drive**: aşağıdaki mount hücresini açın
- **Elden yükleme**: sol paneldeki dosya sekmesinden sürükleyin
- **İndirme**: `!wget ... -P /content/videos/`

Telemetri (`.tlog`) varsa `--telemetry` ile verin; yoksa `agl_m`/`over_sea`/
`sun_elevation` `None` kalır ve o filtreler test edilemez (`vehicle_count`
üzerinden test edin).

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')

import pathlib, shutil
from scripts.register_video import validate_video, InvalidVideoError

VIDEO_DIR = pathlib.Path('/content/videos')
VIDEO_DIR.mkdir(exist_ok=True)

candidates = sorted(p for p in VIDEO_DIR.iterdir()
                    if p.suffix.lower() in ('.mp4', '.mov', '.mkv', '.avi', '.ts'))
assert candidates, 'Once /content/videos/ altina video koyun'

# Her dosyayi ONCEDEN dogrula. Bozuk bir dosya kayit sirasinda degil, uc adim
# sonra proxy uretiminde "moov atom not found" olarak patliyordu - saatlerce
# suren bir toplu yuklemenin ortasinda bunu ogrenmek istemezsiniz.
videos, bozuk = [], []
for p in candidates:
    try:
        info = validate_video(str(p))
        videos.append(p)
        dur = f"{info['duration_s']:.1f}s" if info.get('duration_s') else '?'
        print(f"  OK    {p.name:40s} {info['size_bytes']/1024**2:8.1f} MB  "
              f"{dur:>8s}  {info.get('codec','?')}")
    except InvalidVideoError as e:
        bozuk.append(p)
        print(f"  BOZUK {p.name}")
        print(f"        {str(e).splitlines()[1].strip()}")

total_h = sum(validate_video(str(p)).get('duration_s') or 0 for p in videos) / 3600
free_gb = shutil.disk_usage('/content').free / 1024**3
print(f"\n{len(videos)} saglam video (~{total_h:.2f} saat, ~{total_h*450:.0f} pencere)")
print(f"Bos disk: {free_gb:.1f} GB  |  gereken ~{total_h*0.36 + total_h*0.36:.1f} GB "
      f"(kopya + proxy)")

if bozuk:
    print(f"\n!! {len(bozuk)} bozuk dosya atlanacak. En sik sebep yukleme yarim "
          f"kalmasi -\n   dosya boyutlarini kaynakla karsilastirin ve yeniden yukleyin.")
assert videos, 'Hicbir saglam video yok - yuklemeleri kontrol edin'

## 6. KALİBRASYON — tek video

**Toplu yüklemeden önce mutlaka bu.** Çıktının sonundaki
`Embedding suresi: ... (N.NNx gercek-zaman)` bu denemenin en değerli sayısı:
proje-ozeti.md §8 burada 40x varsayıyor ve doğrulanmadı.

In [ ]:
import time
from scripts.register_video import register
from scripts.ingest_video import run_local

# SUREC ICI calistiriyoruz (`!python -m ...` degil): boylece model bir kez
# yuklenir ve sonraki hucrelerde (batch ayari, toplu yukleme, sorgular)
# tekrar tekrar yuklenmez. Alt surec kullanmak video basina ~1 dk bosa
# harcanmasi demekti.

first = videos[0]
vid = first.stem.replace(' ', '_')

register(vid, str(first))
await run_local(vid, f'{vid}/raw{first.suffix}', None, 'unknown',
                skip_caption=True)

### Batch ayarı

`EMBEDDING_BATCH_SIZE` throughput'un en büyük belirleyicisi. T4 16GB'de
8 → 16 → 24 deneyin; `CUDA out of memory` alırsanız bir kademe düşün.
Model fp16'da ~4,3 GB.

In [ ]:
# Batch degistirip AYNI videoyu tekrar olcun. set_env kullanmak sart -
# dogrudan os.environ'a yazmak surec-ici cagrilarda etkisiz kalir.
set_env(EMBEDDING_BATCH_SIZE=16)
print('Yeni batch:', config.EMBEDDING_BATCH_SIZE)

# Sadece embedding'i olcuyoruz - YOLO ve caption'i atliyoruz.
await run_local(vid, f'{vid}/raw{first.suffix}', None, 'unknown',
                skip_visual=True, skip_caption=True)

## 7. Toplu yükleme

Kalibrasyondan çıkan hıza göre kaç video sığdırabileceğinize karar verin.
Colab oturumu kopabilir — kısa tutun, gerekirse tekrar çalıştırın
(yazım idempotent, aynı pencere iki kez yazılmaz).

In [ ]:
import time
from scripts.register_video import register
from scripts.ingest_video import run_local

# SUREC ICI calistiriyoruz: her `!python -m ...` cagrisi modeli sifirdan
# yukler (~1 dk). 10 videoda bu 10 dakika bosa gider. Burada model bir kez
# yuklenip tum videolarda tekrar kullanilir.

N = 10   # kalibrasyona gore ayarlayin
started = time.time()

for i, path in enumerate(videos[:N], 1):
    v = path.stem.replace(' ', '_')
    print(f'\n=== [{i}/{min(N, len(videos))}] {v} ===')
    register(v, str(path))
    await run_local(v, f'{v}/raw{path.suffix}', None, 'unknown',
                    skip_caption=True)

print(f'\n=== TOPLAM: {time.time()-started:.0f}s ===')

In [ ]:
from common.qdrant_store import get_client

# get_client() onbelleklidir - gomulu Qdrant dosya kilidi kullandigi icin
# ayni dizine ikinci istemci acmak hata verirdi. close() CAGIRMAYIN,
# sonraki hucreler ayni istemciyi kullanacak.
info = get_client().get_collection(config.QDRANT_COLLECTION)
print(f'{info.points_count} pencere yazildi')
print(f'~{info.points_count * config.WINDOW_S / 3600:.2f} saatlik video karsiligi')

## 8. Sorgu testleri

vLLM olmadan yapısal ayrıştırma yok — sorgu tamamen semantiğe düşer.
Yapısal/gevşetme testleri için 9. bölüm.

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

PROMPTS = [
    'a boat moving fast on open water',
    'people swimming near the shore',
    # Zor-negatif cifti: gorsel olarak neredeyse ayni, anlamca zit.
    # IKISI DE AYNI sonucu donduruyorsa model bu ayrimi yapamiyor (§5).
    'a boat approaching the shore at sunset',
    'a boat approaching the shore at sunrise',
]

results = {}
for p in PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    results[p] = run_query(p, top_k=10)
    render(results[p])
    print()

# Zor-negatif karsilastirmasi
a, b = results[PROMPTS[2]], results[PROMPTS[3]]
top_a = [(i.video_id, round(i.t_start)) for i in a.intervals[:3]]
top_b = [(i.video_id, round(i.t_start)) for i in b.intervals[:3]]
print('=' * 70)
print('ZOR-NEGATIF (sunset vs sunrise)')
print('  sunset top-3 :', top_a)
print('  sunrise top-3:', top_b)
print('  SONUC:', 'AYNI - model bu ayrimi YAPAMIYOR' if top_a == top_b
      else 'FARKLI - model ayrimi yapabiliyor')

### Yapısal filtre + gevşetme (vLLM'siz)

`ParsedQuery`'yi elle kurarak vLLM olmadan da yapısal yolu test edebiliriz.
`vehicle_count` YOLO'dan geliyor, telemetriye bağlı değil.

> **Küçük korpusta gevşetme her zaman tetiklenir** — bu bir hata değil.
> Eşik `SEARCH_MIN_RESULTS=5`; korpusta 5'ten az pencere varsa hiçbir filtre
> yeterli sonuç bulamaz ve merdiven sonuna kadar iner. Gevşetmenin *ayırt
> edici* çalıştığını görmek için en az birkaç yüz pencere gerekir
> (≈1 saat video). Az veriyle test ediyorsanız `SEARCH_MIN_RESULTS`'ı
> `set_env(SEARCH_MIN_RESULTS=1)` ile düşürün.

In [ ]:
from query.llm_parser import ParsedQuery, StructuredFilters
from query.hybrid_search import search
from query.interval_merge import merge_matches

for min_v in (2, 50):   # 50 = kasitli imkansiz -> gevsetme tetiklenmeli
    p = ParsedQuery(filters=StructuredFilters(min_vehicle_count=min_v),
                    semantic_text='boats on the water', raw_query='test')
    r = search(p, top_k=10)
    iv = merge_matches(r.matches)
    print(f'--- min_vehicle_count>={min_v} ---')
    print(f'  {len(r.matches)} eslesme -> {len(iv)} aralik | '
          f'gevsetildi={r.was_relaxed} dusen={r.relaxed_fields or "(yok)"}')
    print(f'  embed={r.embed_ms:.0f}ms qdrant={r.qdrant_ms:.0f}ms '
          f'merdiven={r.ladder_steps} adim')
    for i in iv[:3]:
        print(f'    {i.video_id} {i.t_start:.0f}-{i.t_end:.0f}s '
              f'skor={i.score:.3f} tam_eslesme={i.exact_filter_match}')

## 9. vLLM — yapısal ayrıştırma (isteğe bağlı, ağır)

**Bu hat hiç test edilmedi** — projenin en büyük doğrulanmamış parçası.

T4 16GB'de embedding modeli (~4,3 GB) + 7B-AWQ (~5 GB) birlikte sığar ama
sıkışıktır. `--gpu-memory-utilization`'ı düşük tutun. Sunucunun ayağa
kalkması birkaç dakika sürer.

Bakılacak: `Yapisal filtre:` satırı — "gece" yazınca `is_night=true` çıkıyor
mu, yoksa alakasız alan mı doluyor? Ve `gecikme: parse=...ms` — vLLM'in
gerçek maliyeti.

In [ ]:
# SURUM GECMISI (2026-07, gercekten yasandi): once --torch-backend=auto tek
# basina denendi (libcudart.so.13 verdi), sonra vllm==0.8.3'e sabitlendi
# (torch==2.6.0'a bagimli, o surum PyPI'dan KALDIRILMIS - resolver
# tutarsiz davrandi). PyPI JSON API ile dogrulandi: v0.20.0'dan itibaren
# TUM vLLM surumleri torch==2.11.0'a sabit ve bu surum hala kurulabilir.
# Eski surume sabitlemek yerine GUNCEL vLLM'i ihtiyaci olan torch
# surumuyle birlikte ACIKCA istiyoruz.
!pip install -q uv
!uv pip install -q --system "torch==2.11.0" vllm xgrammar --torch-backend=auto 2>&1 | tail -20

# DOGRULAMA - tahmin etmiyoruz, GORUYORUZ: torch GERCEKTEN 2.11.0 mu?
import subprocess
result = subprocess.run(['python3', '-c',
    'import torch, vllm; print(f"TORCH: {torch.__version__}"); print(f"VLLM: {vllm.__version__}")'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('import edilemedi:', result.stderr[-500:])
elif '2.11.0' not in result.stdout.split('VLLM')[0]:
    print('*** UYARI: torch 2.11.0 DEGIL - sabitleme etkisiz kaldi. ***')
    print('Cekirdegi yeniden baslatip bu hucreyi tekrar calistirin.')

In [ ]:
import time
from common.llm import health_check

# Cikti DEVNULL'a DEGIL dosyaya: vLLM sessizce olurse (VRAM yetmezse ya da
# CUDA/kutuphane uyumsuzlugu en sik iki sebep) nedenini gorebilmeliyiz.
LOG = '/content/vllm.log'
vllm_log = open(LOG, 'w')
import subprocess as sp
vllm_proc = sp.Popen([
    'vllm', 'serve', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
    '--structured-outputs-config.backend', 'xgrammar',
    '--gpu-memory-utilization', '0.45',
    '--max-model-len', '2048',
    '--dtype', 'half',   # T4 (compute 7.5) bfloat16 desteklemiyor, half=fp16
], stdout=vllm_log, stderr=sp.STDOUT)
print(f'vLLM baslatiliyor (pid={vllm_proc.pid}) - birkac dakika surer')
print(f'Log: {LOG}   ->  !tail -30 {LOG}')

for i in range(60):
    if vllm_proc.poll() is not None:
        log_tail = open(LOG, encoding='utf-8', errors='replace').read()[-3000:]
        print(f'\nvLLM SUREC OLDU (cikis kodu {vllm_proc.returncode}). Log sonu:')
        print(log_tail)
        if 'libcudart.so' in log_tail or 'undefined symbol' in log_tail:
            print('\n*** torch==2.11.0 sabitlemesine ragmen CUDA/ABI hatasi devam ediyor. ***')
            print('Bu, vLLM #43435 ("not planned") sorununun torch versiyonundan')
            print('BAGIMSIZ, gercekten cozulmemis bir CUDA13 runtime eksikligi oldugunu')
            print('gosterir. Deneyebilecekleriniz:')
            print('  1. nvidia-smi CUDA surumunu kontrol edip elle eslesen backend verin:')
            print('       !uv pip install -q --system "torch==2.11.0" vllm --torch-backend=cu124')
            print('  2. Hedef makineniz (4060, Linux+Docker) icin docker-compose.yml')
            print('     "gpu" profili farkli bir yol - kendi CUDA runtime\'ini tasiyan')
            print('     Docker imaji bu sinif soruna hic girmeyebilir.')
            print('  Sonucu paylasin, requirements-serving.txt buna gore guncellenmeli.')
        break
    if health_check():
        print(f'vLLM hazir ({i*10}s)')
        break
    time.sleep(10)
else:
    print('vLLM 10 dakikada acilmadi. Log sonu:')
    !tail -25 {LOG}
    print('\nEn sik sebep: VRAM yetmiyor. --gpu-memory-utilization dusurun '
          'ya da embedding modelini bosaltip tekrar deneyin.')

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

STRUCTURAL_PROMPTS = [
    'en az 3 tekne gorunen kayitlar',
    'gece deniz uzerinde hareket eden tekne',
    '50 metreden yuksekte ucan arac',
    '20 tekne olan goruntuler',        # imkansiz -> gevsetme tetiklenmeli
    'a boat on the water',             # tamamen semantik -> filtre CIKMAMALI
]

for p in STRUCTURAL_PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    render(run_query(p))
    print()

print('=' * 70)
print('BAKILACAK:')
print('  1. "Yapisal filtre:" satiri dogru alanlari mi doldurdu?')
print('     - "gece" -> is_night=True olmali')
print('     - "3 tekne" -> min_vehicle_count=3 olmali')
print('     - son sorgu (tamamen semantik) -> "filtre yok" olmali.')
print('       Burada bos yere filtre uretiliyorsa sonuclari eler (§8 eki).')
print('  2. "gecikme: parse=...ms" -> vLLM ayristirmanin GERCEK maliyeti.')
print('     vLLM kapaliyken bu deger ~15ms idi (sadece geri cekilme yolu).')

## 10. Sonuçları kaydedin

Denemeden sonra şunları not edin — proje-ozeti.md §8 bunları bekliyor:

1. **Embedding gerçek-zaman katı** (batch değeriyle birlikte) — §8'in 40x
   varsayımı ne kadar uzak?
2. **`parse=...ms`** — vLLM yapısal ayrıştırmanın gerçek gecikmesi. §8'in
   300ms tahmini hiç ölçülmedi.
3. **Zor-negatif sonucu** — "sunset" ve "sunrise" farklı sonuç verdi mi?
   §5'in (model kalitesi) açık kalan kısmına ilk gerçek veri.
4. **Yapısal ayrıştırma doğruluğu** — LLM alanları doğru mu dolduruyor?

> Gecikme rakamlarını Qdrant **gömülü** modda ölçtüyseniz not düşün: o mod
> tam arama yapıyor, gerçek HNSW değil. Sunucu modu için Qdrant ikilisini
> indirip çalıştırın:
> ```
> !wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-gnu.tar.gz
> !tar xzf qdrant-*.tar.gz && (./qdrant &)
> ```
> sonra `QDRANT_LOCAL_PATH` ortam değişkenini silin.